# Clase 12 · Segmentación RFM: los mejores, los que se van y los ocasionales

**Machine Learning** · Programa de Especialización Data Scientist · Sesión 13 de 18

Docente: **Josef Renato Rodriguez**

---

## La pregunta de hoy

En la Clase 10 segmentamos a los clientes por **cómo** usan la tarjeta. Hoy el área comercial pregunta algo más directo:

> **¿Quiénes son nuestros mejores clientes, quiénes se nos están yendo y quiénes casi no compran?**

La herramienta clásica para esto es **RFM**, que mira a cada cliente con tres números:

| Letra | Pregunta | Cómo se mide |
|---|---|---|
| **R**ecencia | ¿Hace cuánto compró por última vez? | días desde la última compra hasta la fecha de corte |
| **F**recuencia | ¿Cuántas veces compra? | número de compras en la ventana |
| **M**onto | ¿Cuánto gasta? | suma de sus compras en la ventana |

Es simple, se explica en una reunión y funciona. Hoy lo construimos **desde el transaccional**, como se hace en un banco:
casi 440 mil compras con tarjeta de los mismos clientes de las Clases 9 y 10.

## Qué vas a hacer hoy

| Bloque | Contenido | Min |
|---|---|---|
| A | Del transaccional al RFM: **fecha de corte** y fuga de información | 25 |
| B | Puntajes por **quintiles**, el problema de los empates y el **mapa de segmentos** | 30 |
| C | **K-Means sobre RFM**: ¿los datos encuentran los mismos segmentos que las reglas? | 25 |
| D | **Validar con el futuro**, cruzar con el default y decidir qué hacer con cada segmento | 30 |

---
## Celda 0 · Librerías y datos

La celda lee tres archivos del repositorio del curso ([Super_ML_course](https://github.com/josefrodrim/Super_ML_course)):

- `transacciones_tarjetas.csv.gz` (carpeta `Clase 12/Data`): **una fila por compra**, de julio de 2025 a septiembre de 2026.
  Viene comprimido; `pandas` lo lee igual.
- `segmentacion_tarjetas.csv` (carpeta `Clase 10/Data`): solo la usamos como **lista de clientes** (los 6 685).
- `cartera_tarjetas.csv` (carpeta `Clase_9/Datos`): de aquí sale el **default** de la última foto de 2026, para el Bloque D.

El diccionario de variables está en `Clase 12/Data/diccionario_transacciones_tarjetas.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
COLORES = [BLUE, MAG, GREEN, "#F59F00", "#7048E8", GRAY, NAVY, "#0CA678"]
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

# Gráficos interactivos: pasa el mouse para ver valores, arrastra para hacer zoom, doble clic para volver.
pio.templates["curso"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Arial, sans-serif", size=13, color=NAVY),
    colorway=COLORES, hoverlabel=dict(font_size=12),
    margin=dict(l=60, r=30, t=70, b=50)))
pio.templates.default = "plotly_white+curso"

ORDEN_SEG = ["Campeones", "Leales", "Potenciales", "Ocasionales", "En riesgo", "Enfriándose", "Perdidos"]
COLOR_SEG = dict(zip(ORDEN_SEG, [NAVY, BLUE, "#4DABF7", "#A5B4CB", MAG, "#F59F00", GRAY]))
COLOR_CLUSTER = {"Alto valor": NAVY, "Activos": BLUE, "Enfriándose": "#F59F00", "Dormidos": MAG}

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

URL_TX = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/main/Clase_12/Data/transacciones_tarjetas.csv.gz"
URL_SEG = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/main/Clase%2010/Data/segmentacion_tarjetas.csv"
URL_CARTERA = "https://raw.githubusercontent.com/josefrodrim/Super_ML_course/main/Clase_9/Datos/cartera_tarjetas.csv"

tx = pd.read_csv(URL_TX, compression="gzip", parse_dates=["fecha_hora"])
clientes = pd.read_csv(URL_SEG, usecols=["id_cliente"])["id_cliente"]
cartera = pd.read_csv(URL_CARTERA)

# default de la última foto de 2026 de cada cliente (lo mismo que hicimos en la Clase 10)
default = (cartera[cartera["periodo"] >= "2026-01"].sort_values("periodo")
           .groupby("id_cliente").tail(1).set_index("id_cliente")["default_3m"])

CORTE = pd.Timestamp("2026-06-30")       # la foto se toma al cierre de este día
print(f"{len(tx):,} compras · {tx['id_cliente'].nunique():,} clientes con al menos una compra · {len(clientes):,} clientes en la base".replace(",", " "))
print("desde", tx["fecha_hora"].min(), "hasta", tx["fecha_hora"].max())
tx.head()

---
# Bloque A · Del transaccional al RFM  ·  25 min

### Demostración A.0 — Cómo se ve un año de compras

Antes de calcular nada, miramos las compras mes a mes. Fíjate en diciembre y en julio (gratificaciones y fiestas).

In [ ]:
# ── DEMOSTRACIÓN A.0 ──────────────────────────────────────────────────
por_mes = tx.groupby(tx["fecha_hora"].dt.to_period("M")).agg(compras=("id_transaccion", "size"), monto=("monto", "sum"))
por_mes.index = por_mes.index.to_timestamp()
por_mes["etapa"] = np.where(por_mes.index <= CORTE, "Historia (entra al RFM)", "Futuro (solo para validar)")

fig = make_subplots(specs=[[{"secondary_y": True}]])
for etapa, color in [("Historia (entra al RFM)", BLUE), ("Futuro (solo para validar)", GRAY)]:
    d = por_mes[por_mes["etapa"] == etapa]
    fig.add_bar(x=d.index, y=d["monto"] / 1e6, name=etapa, marker_color=color,
                customdata=d[["compras"]], secondary_y=False,
                hovertemplate="<b>%{x|%b %Y}</b><br>Monto: S/ %{y:.2f} M<br>Compras: %{customdata[0]:,}<extra></extra>")
fig.add_scatter(x=por_mes.index, y=por_mes["compras"], name="N.º de compras", mode="lines+markers",
                line=dict(color=MAG, width=2), secondary_y=True,
                hovertemplate="%{y:,} compras<extra></extra>")
fig.add_vline(x=CORTE + pd.Timedelta(days=15), line_dash="dash", line_color=NAVY)
fig.add_annotation(x=CORTE + pd.Timedelta(days=15), y=1.04, yref="paper", text="Fecha de corte · 30 jun 2026",
                   showarrow=False, font=dict(color=NAVY, size=12))
fig.update_layout(title="Compras por mes: la historia hasta el corte y lo que pasó después",
                  height=430, hovermode="x unified", legend=dict(orientation="h", y=-0.18),
                  xaxis=dict(dtick="M1", tickformat="%b<br>%Y"))
fig.update_yaxes(title_text="Monto (millones de S/)", secondary_y=False)
fig.update_yaxes(title_text="N.º de compras", secondary_y=True, showgrid=False, tickformat=",.0f")
fig.show()

canal = tx["canal"].value_counts(normalize=True).rename_axis("canal").reset_index(name="share")
fig = px.bar(canal, x="share", y="canal", orientation="h", text="share", color_discrete_sequence=[BLUE],
             title="¿Por qué canal compran?")
fig.update_traces(texttemplate="%{text:.1%}", hovertemplate="%{y}: %{x:.1%}<extra></extra>")
fig.update_layout(height=300, xaxis=dict(tickformat=".0%", title=""), yaxis=dict(title="", categoryorder="total ascending"))
fig.show()

### La fecha de corte

Un RFM es una **foto**: se toma en una fecha y solo puede usar lo que pasó **hasta** esa fecha. Hoy la foto es al
**30 de junio de 2026**, la misma fecha de la última foto de la cartera de la Clase 9. Así podemos cruzar con el default de
los tres meses siguientes **sin hacer trampa**.

Los datos llegan hasta septiembre de 2026 a propósito: julio a septiembre es **el futuro** de la foto. No entra al RFM;
lo vamos a usar en el Bloque D para comprobar si los segmentos acertaron.

### Ejercicio A.1 — Separar la historia del futuro

1. `historia`: las compras con `fecha_hora` **hasta el cierre** del 30 de junio de 2026. Cuidado: `fecha_hora` tiene hora,
   así que «menor o igual que `CORTE`» deja fuera casi todo el día 30. Usa `< CORTE + pd.Timedelta(days=1)`.
2. `futuro`: el resto (julio a septiembre de 2026).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
historia = None   # compras hasta el cierre del 30/06/2026
futuro = None     # compras desde el 01/07/2026

In [ ]:
# ── VERIFICACIÓN A.1 ──────────────────────────────────────────────────
r = [check("compras en la historia", None if historia is None else len(historia), 355704, tol=0),
     check("compras en el futuro", None if futuro is None else len(futuro), 83321, tol=0),
     check_bool("la última compra de la historia es del 30 de junio",
                historia is not None and historia["fecha_hora"].max().date() == CORTE.date(),
                "¿usaste <= CORTE? Así se pierde casi todo el 30 de junio")]
print()
print("A.1 OK" if all(r) else "Revisa A.1")

### Ejercicio A.2 — La tabla RFM

Una fila por cliente, con índice `id_cliente` y tres columnas:

- `recencia`: días entre la **fecha** de la última compra y `CORTE`. Una compra el mismo 30 de junio da 0.
  (Pista: `(CORTE - ultima.dt.normalize()).dt.days`.)
- `frecuencia`: número de compras en `historia`.
- `monto`: suma de `monto` en `historia`.

**La trampa:** un `groupby` sobre las transacciones **solo ve a los clientes que compraron**. Los que no compraron nada en
el año desaparecen, y son justamente los que más interesan. Parte de la lista `clientes` (los 6 685) y completa a los que no
tienen compras con `frecuencia = 0`, `monto = 0` y `recencia = 366` (más días que la ventana).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
agg = None    # historia agrupada por id_cliente: última fecha, número de compras y suma del monto
rfm = None    # una fila por cada uno de los 6 685 clientes: recencia, frecuencia, monto

In [ ]:
# ── VERIFICACIÓN A.2 ──────────────────────────────────────────────────
r = [check("clientes en la tabla", None if rfm is None else len(rfm), 6685, tol=0),
     check("clientes sin ninguna compra", None if rfm is None else (rfm["frecuencia"] == 0).sum(), 320, tol=0),
     check("mediana de la recencia (días)", None if rfm is None else rfm["recencia"].median(), 7, tol=0),
     check("monto total del año (millones de S/)", None if rfm is None else rfm["monto"].sum() / 1e6, 212.0514, tol=0.001)]
print()
print("A.2 OK" if all(r) else "Revisa A.2")

**La mitad de la cartera compró en los últimos 7 días.** Es una cartera muy activa: eso va a importar en el Bloque B.

### Demostración A.3 — ¿Qué pasa si te olvidas de la fecha de corte?

Calculamos la recencia de dos maneras y medimos qué tan bien **separa a los que caen en default** (con el AUC de la
Clase 6: 0.5 es azar, 1 es perfecto):

- **Bien:** con la historia, al 30 de junio.
- **Mal:** con todas las compras, al 30 de septiembre. Es decir, mirando los tres meses en los que se mide el default.

In [ ]:
# ── DEMOSTRACIÓN A.3 ──────────────────────────────────────────────────
if rfm is not None:
    fin = pd.Timestamp("2026-09-30")
    ult_todo = tx.groupby("id_cliente")["fecha_hora"].max().reindex(clientes)
    rec_fuga = (fin - ult_todo.dt.normalize()).dt.days.fillna(458)
    y = default.reindex(clientes)
    print(f"AUC de la recencia con la fecha de corte correcta: {roc_auc_score(y, rfm['recencia']):.3f}")
    print(f"AUC de la recencia mirando el futuro:              {roc_auc_score(y, rec_fuga):.3f}")

La recencia «mal hecha» parece **mucho mejor** (0.89 contra 0.74). No es que sea mejor: un cliente que cae en default tiene
la tarjeta **bloqueada**, así que deja de comprar en julio. Si la recencia mira julio a septiembre, está viendo la respuesta.

> **Fuga de información (leakage).** Es el mismo error de la Clase 6, ahora en una variable de comportamiento. En producción
> nadie tiene los datos del futuro, y el modelo que parecía excelente se cae. **Toda variable se calcula con lo que había a la
> fecha de corte.**

---
# Bloque B · Puntajes y segmentos  ·  30 min

El RFM clásico convierte cada número en un **puntaje de 1 a 5** con **quintiles**: el 20 % con menos compras recibe 1, el
siguiente 20 % recibe 2, y así. En recencia es al revés: **menos días es mejor**, así que los más recientes reciben 5.

Es la misma lógica de las bandas de la Clase 9 (de Muy alto a Muy bajo).

### Ejercicio B.1 — Puntajes F y M por quintiles

Crea `rfm["F"]` y `rfm["M"]` con `pd.qcut(..., 5, labels=[1, 2, 3, 4, 5])`.

**Detalle importante:** la frecuencia es un **conteo** y hay muchos empates (cientos de clientes con exactamente 30 compras).
`pd.qcut` necesita cortes distintos, y con empates falla o arma grupos desiguales. El truco estándar es cortar sobre el
**ranking**: `rfm["frecuencia"].rank(method="first")`, que rompe los empates por orden de aparición. Usa el mismo truco
en el monto. Convierte el resultado a `int`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# rfm["F"] = ...
# rfm["M"] = ...

In [ ]:
# ── VERIFICACIÓN B.1 ──────────────────────────────────────────────────
tiene = rfm is not None and {"F", "M"} <= set(rfm.columns)
r = [check_bool("cada puntaje de F tiene 1 337 clientes", tiene and set(rfm["F"].value_counts()) == {1337}),
     check("menor frecuencia con F = 5", rfm.loc[rfm["F"] == 5, "frecuencia"].min() if tiene else None, 81, tol=0),
     check("menor monto con M = 5 (S/)", rfm.loc[rfm["M"] == 5, "monto"].min() if tiene else None, 34796.13, tol=0.01),
     check("correlación de Spearman entre F y M", rfm[["frecuencia", "monto"]].corr("spearman").iloc[0, 1] if tiene else None, 0.539, tol=0.005)]
print()
print("B.1 OK" if all(r) else "Revisa B.1")

F y M tienen una correlación de **0.54**: el que compra seguido suele gastar más, pero no siempre (un cliente que compra
pasajes dos veces al año gasta mucho con poca frecuencia). Por eso las juntamos en un solo eje de **valor**, sin perder
ninguna de las dos.

### Demostración B.2 — Los quintiles no sirven para la recencia de esta cartera

In [ ]:
# ── DEMOSTRACIÓN B.2 ──────────────────────────────────────────────────
if rfm is not None:
    cortes = rfm["recencia"].quantile([0, .2, .4, .6, .8, 1]).values
    print("Cortes de los quintiles de recencia (días):", cortes)
    print("\nCon quintiles, un cliente que compró hace 13 días recibiría R = 2 (de 5),")
    print("el mismo puntaje bajo que uno que no compra hace casi dos meses.")

    def lineas(valores, color, prefijo):
        shapes = [dict(type="line", x0=v, x1=v, y0=0, y1=1, yref="paper", line=dict(color=color, dash="dash", width=1.5))
                  for v in valores]
        # etiquetas escalonadas para que no se encimen cuando los cortes están muy juntos
        notas = [dict(x=v, y=0.97 - 0.08 * (k % 3), yref="paper", text=f"{prefijo}{v:g} d", showarrow=False,
                      xanchor="left", xshift=3, bgcolor="rgba(255,255,255,0.8)",
                      font=dict(color=color, size=11)) for k, v in enumerate(valores)]
        return shapes, notas

    sh_q, an_q = lineas(cortes[1:-1], MAG, "")
    sh_n, an_n = lineas([7, 30, 60, 120], GREEN, "")

    fig = px.histogram(rfm, x="recencia", color_discrete_sequence=[BLUE])
    fig.update_traces(xbins=dict(start=0, end=370, size=7),
                      hovertemplate="Recencia %{x} días<br>%{y:,} clientes<extra></extra>")
    fig.update_layout(
        title="Recencia: casi todo el peso en la primera semana",
        xaxis_title="días desde la última compra", yaxis_title="clientes", height=420, bargap=0.05,
        shapes=sh_q, annotations=an_q,
        updatemenus=[dict(type="buttons", direction="right", x=1, xanchor="right", y=1.18, showactive=True,
                          buttons=[dict(label="Cortes por quintiles", method="relayout",
                                        args=[{"shapes": sh_q, "annotations": an_q}]),
                                   dict(label="Cortes de negocio (B.3)", method="relayout",
                                        args=[{"shapes": sh_n, "annotations": an_n}]),
                                   dict(label="Sin cortes", method="relayout",
                                        args=[{"shapes": [], "annotations": []}])])])
    fig.show()

Los quintiles cortan en **1, 4, 12 y 55 días**. Tres de los cinco puntajes separan a clientes que compraron **en el mismo
mes**: para el negocio son igual de activos. Los quintiles funcionan cuando el dato está repartido; aquí la mitad de la
cartera compró la última semana.

La solución habitual en banca es usar **cortes de negocio** para R y dejar los quintiles para F y M:

| R | Días desde la última compra | Lectura |
|---|---|---|
| 5 | 0 a 7 | compró esta semana |
| 4 | 8 a 30 | compró este mes |
| 3 | 31 a 60 | lleva un mes sin comprar |
| 2 | 61 a 120 | dos a cuatro meses sin comprar |
| 1 | más de 120, o ninguna compra | inactivo |

### Ejercicio B.3 — Puntaje R con cortes de negocio

Crea `rfm["R"]` con `pd.cut` y los bordes `[-1, 7, 30, 60, 120, 10**6]`. Ojo con el orden de las etiquetas: el primer
tramo (0 a 7 días) es el **5**. Convierte a `int`. Después crea `rfm["FM"]`, el **promedio** de F y M.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# rfm["R"] = ...
# rfm["FM"] = ...

In [ ]:
# ── VERIFICACIÓN B.3 ──────────────────────────────────────────────────
tiene = rfm is not None and {"R", "FM"} <= set(rfm.columns)
cuenta = rfm["R"].value_counts() if tiene else None
r = [check("clientes con R = 5", None if cuenta is None else cuenta.get(5), 3408, tol=0),
     check("clientes con R = 1", None if cuenta is None else cuenta.get(1), 818, tol=0),
     check("FM promedio", rfm["FM"].mean() if tiene else None, 3.0, tol=1e-9)]
print()
print("B.3 OK" if all(r) else "Revisa B.3")

### Ejercicio B.4 — El mapa de segmentos

Con dos ejes, **R** (qué tan reciente) y **FM** (cuánto vale), cada cliente cae en un segmento. Las reglas se revisan **en
este orden** y gana la primera que se cumple:

| # | Segmento | Regla | Quiénes son |
|---|---|---|---|
| 1 | **Campeones** | R = 5 y FM ≥ 4.5 | compran esta semana, mucho y seguido |
| 2 | **Leales** | R ≥ 4 y FM ≥ 3.5 | activos y de alto valor |
| 3 | **Potenciales** | R ≥ 4 y FM ≥ 2.5 | activos, valor medio: se pueden desarrollar |
| 4 | **Ocasionales** | R ≥ 4 (y FM < 2.5) | compran, pero poco |
| 5 | **En riesgo** | FM ≥ 3 (y R ≤ 3) | **valían** y dejaron de comprar |
| 6 | **Enfriándose** | R = 3 (y FM < 3) | valor bajo y un mes sin comprar |
| 7 | **Perdidos** | el resto | valor bajo y meses sin comprar |

Completa la función `segmentar(r, fm)` y crea `rfm["segmento"]`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def segmentar(r, fm):
    if r == 5 and fm >= 4.5:
        return "Campeones"
    if r >= 4 and fm >= 3.5:
        return "Leales"
    # completa las otras cinco reglas, en orden
    return None

# rfm["segmento"] = [segmentar(r, fm) for r, fm in zip(rfm["R"], rfm["FM"])]

In [ ]:
# ── VERIFICACIÓN B.4 ──────────────────────────────────────────────────
tam = rfm["segmento"].value_counts() if rfm is not None and "segmento" in rfm else None
def t(nombre):
    return None if tam is None or nombre not in tam.index else tam[nombre]
r = [check_bool("hay 7 segmentos y ningún cliente sin segmento", tam is not None and len(tam) == 7 and rfm["segmento"].notna().all(),
                "¿completaste las cinco reglas? ¿la última devuelve 'Perdidos'?"),
     check("Campeones", t("Campeones"), 896, tol=0),
     check("En riesgo", t("En riesgo"), 436, tol=0),
     check("Perdidos", t("Perdidos"), 1038, tol=0)]
print()
print("B.4 OK" if all(r) else "Revisa B.4")

### Demostración B.5 — Quiénes son los mejores: el mapa y la concentración del gasto

In [ ]:
# ── DEMOSTRACIÓN B.5 ──────────────────────────────────────────────────
if rfm is not None and "segmento" in rfm:
    mapa = pd.crosstab(rfm["R"], rfm["FM"]).sort_index(ascending=False)
    seg_celda = [[segmentar(r, fm) for fm in mapa.columns] for r in mapa.index]
    fig = go.Figure(go.Heatmap(
        z=mapa.values, x=[f"{c:g}" for c in mapa.columns], y=[f"R = {r}" for r in mapa.index],
        colorscale="Blues", customdata=seg_celda, text=mapa.values, texttemplate="%{text}",
        colorbar=dict(title="clientes"),
        hovertemplate="%{y} · FM = %{x}<br><b>%{customdata}</b><br>%{z:,} clientes<extra></extra>"))
    fig.update_layout(title="Clientes por recencia y valor · pasa el mouse para ver el segmento de cada casilla",
                      xaxis_title="FM (promedio de F y M)", height=400)
    fig.show()

    valor = rfm.groupby("segmento").agg(clientes=("monto", "size"), monto=("monto", "sum"))
    valor["% clientes"] = (valor["clientes"] / valor["clientes"].sum() * 100).round(1)
    valor["% del gasto"] = (valor["monto"] / valor["monto"].sum() * 100).round(1)
    print(valor.sort_values("% del gasto", ascending=False)[["clientes", "% clientes", "% del gasto"]].to_string())

    largo = (valor.reindex(ORDEN_SEG).reset_index()
             .melt(id_vars=["segmento", "clientes", "monto"], value_vars=["% clientes", "% del gasto"],
                   var_name="medida", value_name="pct"))
    fig = px.bar(largo, x="pct", y="segmento", color="medida", barmode="group", orientation="h", text="pct",
                 color_discrete_map={"% clientes": "#A5B4CB", "% del gasto": BLUE},
                 custom_data=["clientes", "monto"],
                 title="Peso de cada segmento: en clientes vs. en gasto")
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside",
                      hovertemplate="<b>%{y}</b><br>%{x:.1f}%<br>%{customdata[0]:,} clientes · "
                                    "S/ %{customdata[1]:,.0f}<extra></extra>")
    fig.update_layout(height=450, xaxis_title="%", yaxis=dict(title="", categoryorder="array",
                      categoryarray=ORDEN_SEG[::-1]), legend=dict(orientation="h", y=-0.15, title=""))
    fig.show()

    fig = px.treemap(valor.reset_index(), path=["segmento"], values="monto", color="segmento",
                     color_discrete_map=COLOR_SEG, custom_data=["clientes", "% clientes", "% del gasto"],
                     title="¿Dónde está el gasto? · el área es el monto · clic en un segmento para ampliarlo")
    fig.update_traces(texttemplate="<b>%{label}</b><br>%{customdata[2]}% del gasto",
                      hovertemplate="<b>%{label}</b><br>S/ %{value:,.0f}<br>%{customdata[0]:,} clientes "
                                    "(%{customdata[1]}%)<extra></extra>")
    fig.update_layout(height=420, margin=dict(t=60, l=10, r=10, b=10))
    fig.show()

**Campeones y Leales son el 43 % de los clientes y hacen el 78 % del gasto.** Solo los Campeones (13 %) hacen el 46 %.
Es la regla de Pareto que aparece en casi toda cartera: perder un Campeón cuesta lo que perder a decenas de Ocasionales.

En el otro extremo, **Perdidos** son el 16 % de los clientes y apenas el 2 % del gasto.

---
# Bloque C · K-Means sobre RFM  ·  25 min

Las reglas del Bloque B las escribimos **nosotros**. ¿Qué segmentos encuentran **los datos**, si dejamos que K-Means
agrupe con las mismas tres variables? Es lo mismo de la Clase 10, con dos cuidados:

1. **Sin los 320 clientes sin compras.** Ya sabemos qué son; si entran, K-Means gasta un grupo en separarlos (con
   `log1p(0) = 0` quedan aislados) y la silueta premia esa separación obvia.
2. **Logaritmo y escalado.** Los tres son montos y conteos sesgados.

### Ejercicio C.1 — Elegir k

1. `activos`: las filas de `rfm` con `frecuencia > 0`.
2. `Xs`: `np.log1p` de `recencia`, `frecuencia` y `monto` de `activos`, escalado con `StandardScaler`.
3. `sil`: una `pd.Series` con la silueta de K-Means para k de 2 a 8 (índice = k). Usa `n_init=10, random_state=SEED`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
activos = None   # clientes con al menos una compra
Xs = None        # log1p de las tres variables, escalado
sil = None       # silueta para k = 2..8

In [ ]:
# ── VERIFICACIÓN C.1 ──────────────────────────────────────────────────
r = [check("clientes activos", None if activos is None else len(activos), 6365, tol=0),
     check("silueta con k = 2", None if sil is None else sil.get(2), 0.3985, tol=0.01),
     check("silueta con k = 4", None if sil is None else sil.get(4), 0.3216, tol=0.01)]
print()
print("C.1 OK" if all(r) else "Revisa C.1")
if sil is not None:
    fig = go.Figure(go.Scatter(x=sil.index, y=sil.values, mode="lines+markers+text",
                               text=[f"{v:.3f}" for v in sil.values], textposition="top center",
                               marker=dict(size=10, color=[MAG if k == 4 else BLUE for k in sil.index]),
                               line=dict(color=BLUE), hovertemplate="k = %{x}<br>silueta = %{y:.4f}<extra></extra>"))
    fig.add_annotation(x=4, y=sil[4], text="k = 4: la que usamos en C.2", ax=-90, ay=50, font=dict(color=MAG))
    fig.update_layout(title="Silueta de K-Means sobre RFM", xaxis=dict(title="k", dtick=1),
                      yaxis_title="silueta promedio", height=380)
    fig.show()

La silueta prefiere **k = 2**, pero dos grupos («compran mucho» y «compran poco») no le sirven a nadie. Como en la Clase 10,
la métrica propone y el negocio decide: tomamos **k = 4**, el más chico que distingue a los que se están yendo.

### Ejercicio C.2 — Cuatro clusters con nombre

1. `km4`: K-Means con 4 grupos sobre `Xs` (`n_init=10, random_state=SEED`). Guarda las etiquetas en `activos["cluster"]`.
2. `perfil`: la **mediana** de `recencia`, `frecuencia` y `monto` por cluster, más la columna `clientes`.
3. Pon nombres con reglas sobre el perfil, en este orden: **Alto valor** (el de mayor monto), **Dormidos** (de los que
   quedan, el de mayor recencia), **Enfriándose** (de los que quedan, el de mayor recencia) y **Activos** (el último).
   Guarda el nombre en `activos["cluster_nombre"]`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
km4 = None
# activos["cluster"] = ...
perfil = None
# nombres = {...}            # {número de cluster: nombre}
# activos["cluster_nombre"] = activos["cluster"].map(nombres)

In [ ]:
# ── VERIFICACIÓN C.2 ──────────────────────────────────────────────────
tam = activos["cluster_nombre"].value_counts() if activos is not None and "cluster_nombre" in activos else None
def t(nombre):
    return None if tam is None or nombre not in tam.index else tam[nombre]
r = [check("clientes en Alto valor", t("Alto valor"), 1164, tol=30),
     check("clientes en Dormidos", t("Dormidos"), 341, tol=30),
     check("clientes en Enfriándose", t("Enfriándose"), 1998, tol=40),
     check("clientes en Activos", t("Activos"), 2862, tol=40)]
print()
print("C.2 OK" if all(r) else "Revisa C.2")

### Demostración C.3 — ¿Reglas o clusters?

In [ ]:
# ── DEMOSTRACIÓN C.3 ──────────────────────────────────────────────────
if activos is not None and "cluster_nombre" in activos:
    orden = ["Campeones", "Leales", "Potenciales", "Ocasionales", "En riesgo", "Enfriándose", "Perdidos"]
    cols = ["Alto valor", "Activos", "Enfriándose", "Dormidos"]
    cruce = pd.crosstab(activos["segmento"], activos["cluster_nombre"]).reindex(orden)
    print(cruce[cols].to_string())
    print(f"\nARI entre reglas y clusters: {adjusted_rand_score(activos['segmento'], activos['cluster']):.3f}")

    # mapa del cruce: color = % de cada segmento de reglas que cae en cada cluster
    pct = cruce[cols].div(cruce[cols].sum(axis=1), axis=0)
    fig = go.Figure(go.Heatmap(
        z=pct.values, x=cols, y=orden, colorscale="Blues", zmin=0, zmax=1, customdata=cruce[cols].values,
        text=[[f"{p:.0%}" for p in fila] for fila in pct.values], texttemplate="%{text}",
        colorbar=dict(title="% de la fila", tickformat=".0%"),
        hovertemplate="Reglas: <b>%{y}</b><br>K-Means: <b>%{x}</b><br>%{customdata:,} clientes "
                      "(%{z:.1%} del segmento)<extra></extra>"))
    fig.update_layout(title="¿Dónde caen los segmentos de reglas en los clusters de K-Means?",
                      xaxis_title="cluster de K-Means", yaxis=dict(title="segmento por reglas", autorange="reversed"),
                      height=430)
    fig.show()

    # los clientes en el espacio RFM: gira, haz zoom y apaga clusters en la leyenda
    vista = activos.reset_index().rename(columns={"index": "id_cliente"})
    fig = px.scatter_3d(vista, x="recencia", y="frecuencia", z="monto", color="cluster_nombre",
                        color_discrete_map=COLOR_CLUSTER, log_x=True, log_y=True, log_z=True, opacity=0.55,
                        hover_name=vista.columns[0], hover_data={"segmento": True, "cluster_nombre": False,
                                                                 "recencia": True, "frecuencia": True,
                                                                 "monto": ":,.0f"},
                        category_orders={"cluster_nombre": cols},
                        title="Clusters de K-Means en el espacio RFM (escala log) · arrastra para girar")
    fig.update_traces(marker=dict(size=2.5))
    fig.update_layout(height=620, legend=dict(title="cluster", itemsizing="constant"),
                      scene=dict(xaxis_title="recencia (días)", yaxis_title="frecuencia", zaxis_title="monto (S/)"))
    fig.show()

Coinciden en los **extremos**: todos los Campeones caen en Alto valor o Activos, y casi todos los En riesgo en Enfriándose.
En el medio no: K-Means mete en **Enfriándose** a clientes que las reglas reparten entre cinco segmentos. El ARI de **0.25**
lo resume: se parecen, pero no son lo mismo.

| | Reglas RFM | K-Means sobre RFM |
|---|---|---|
| Quién define los grupos | el negocio | los datos |
| Se explica en una reunión | sí, en una frase | hay que mostrar el perfil |
| Mes a mes | el mismo cliente con los mismos números cae en el mismo segmento | los centros se mueven y los grupos cambian |
| Para qué sirve más | campañas y tableros recurrentes | **descubrir** cortes que las reglas no ven |

En la práctica se usan **las dos**: el clustering para explorar y proponer cortes, y las reglas para operar todos los meses.

---
# Bloque D · Validar con el futuro y decidir  ·  30 min

Un segmento vale si **predice algo** que el negocio no sabía. Tenemos dos cosas que pasaron **después** de la foto:

- `futuro`: si el cliente volvió a comprar entre julio y septiembre.
- `default`: si cayó en default en esos tres meses (de la cartera de la Clase 9).

### Ejercicio D.1 — La tabla de validación

1. `rfm["compro_jul_sep"]`: 1 si el cliente tiene al menos una compra en `futuro`, 0 si no.
2. `rfm["default_3m"]`: el default de cada cliente (la serie `default`, alineada por `id_cliente`).
3. `validacion`: por segmento, `clientes`, `compro_jul_sep` (promedio) y `default_3m` (promedio), ordenada de menor a mayor
   `compro_jul_sep`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# rfm["compro_jul_sep"] = ...
# rfm["default_3m"] = ...
validacion = None

In [ ]:
# ── VERIFICACIÓN D.1 ──────────────────────────────────────────────────
def v(seg, col):
    return None if validacion is None or seg not in validacion.index else validacion.loc[seg, col]
r = [check("compraron en jul-sep: Campeones", v("Campeones", "compro_jul_sep"), 0.973, tol=0.002),
     check("compraron en jul-sep: Perdidos", v("Perdidos", "compro_jul_sep"), 0.204, tol=0.002),
     check("default: En riesgo", v("En riesgo", "default_3m"), 0.319, tol=0.002),
     check("default: toda la cartera", None if validacion is None else rfm["default_3m"].mean(), 0.0905, tol=0.001)]
print()
print("D.1 OK" if all(r) else "Revisa D.1")

In [ ]:
# ── DEMOSTRACIÓN D.2 ──────────────────────────────────────────────────
if validacion is not None:
    tasa_cartera = rfm["default_3m"].mean()
    vv = validacion.sort_values("compro_jul_sep")
    vd = validacion.sort_values("default_3m")
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                        subplot_titles=("¿Volvió a comprar en julio a septiembre?",
                                        "Tasa de default a 3 meses · línea: cartera"))
    fig.add_bar(x=vv["compro_jul_sep"], y=vv.index, orientation="h", row=1, col=1,
                marker_color=[COLOR_SEG.get(s, BLUE) for s in vv.index], customdata=vv["clientes"],
                text=vv["compro_jul_sep"], texttemplate="%{text:.0%}", textposition="outside",
                hovertemplate="<b>%{y}</b><br>%{x:.1%} volvió a comprar<br>%{customdata:,} clientes<extra></extra>")
    fig.add_bar(x=vd["default_3m"], y=vd.index, orientation="h", row=1, col=2,
                marker_color=[MAG if x > 0.15 else GRAY for x in vd["default_3m"]], customdata=vd["clientes"],
                text=vd["default_3m"], texttemplate="%{text:.1%}", textposition="outside",
                hovertemplate="<b>%{y}</b><br>default: %{x:.1%}<br>%{customdata:,} clientes<extra></extra>")
    fig.add_vline(x=tasa_cartera, line_dash="dash", line_color=NAVY, row=1, col=2)
    fig.add_annotation(x=tasa_cartera, y=0, yref="y2 domain", xref="x2", text=f"cartera {tasa_cartera:.1%}",
                       showarrow=False, xanchor="left", yanchor="bottom", xshift=4, bgcolor="rgba(255,255,255,0.85)",
                       font=dict(color=NAVY, size=11))
    fig.update_xaxes(range=[0, 1.12], tickformat=".0%", row=1, col=1)
    fig.update_xaxes(range=[0, vd["default_3m"].max() * 1.25], tickformat=".0%", row=1, col=2)
    fig.update_layout(height=430, showlegend=False, margin=dict(t=80))
    fig.show()

    # las dos cosas a la vez: actividad futura vs. riesgo, el tamaño de la burbuja es el número de clientes
    burbuja = validacion.reset_index()
    fig = px.scatter(burbuja, x="compro_jul_sep", y="default_3m", size="clientes", color="segmento", text="segmento",
                     color_discrete_map=COLOR_SEG, size_max=45, category_orders={"segmento": ORDEN_SEG},
                     hover_data={"clientes": ":,", "compro_jul_sep": ":.1%", "default_3m": ":.1%", "segmento": False},
                     labels={"compro_jul_sep": "volvió a comprar (jul-sep)", "default_3m": "default a 3 meses"},
                     title="Actividad futura vs. riesgo por segmento")
    fig.add_hline(y=tasa_cartera, line_dash="dash", line_color=NAVY,
                  annotation_text=f"default de la cartera {tasa_cartera:.1%}", annotation_position="top left")
    # los segmentos activos quedan juntos abajo a la derecha: cada etiqueta va hacia un lado distinto
    lado = {"Campeones": "middle right", "Leales": "top right", "Potenciales": "top left", "Ocasionales": "bottom left"}
    for tr in fig.data:
        tr.textposition = lado.get(tr.name, "top center")
    fig.update_layout(height=480, showlegend=True, legend_title="", xaxis=dict(tickformat=".0%", range=[0, 1.15]),
                      yaxis=dict(tickformat=".0%"))
    fig.show()

Dos lecturas, y la segunda es la importante:

1. **El RFM predice la actividad.** El 97 % de los Campeones volvió a comprar; de los Perdidos, solo el 20 %, y de los
   En riesgo, el 45 %. Con tres números y unas reglas se anticipa quién se va.
2. **En riesgo no siempre significa «se fue a la competencia».** Ese segmento tiene una tasa de default de **31.9 %**, 3.5
   veces la cartera. Muchos dejaron de comprar porque **coparon la línea**: no pueden comprar, y después no pueden pagar.

> **Por eso el RFM no se usa solo.** Si el área comercial lanza una campaña de «te extrañamos, aquí tienes más línea» a
> todo En riesgo, le está ofreciendo crédito a uno de cada tres clientes que va a caer en default. Antes de actuar, se
> cruza con el **score de riesgo** de la Clase 9.

### Qué hacer con cada segmento

| Segmento | % clientes | % gasto | Acción |
|---|---|---|---|
| Campeones | 13 % | 46 % | Cuidarlos: beneficios, atención preferente. **No** saturarlos de ofertas. |
| Leales | 29 % | 33 % | Programas de puntos, subir a Campeones con ofertas de categorías nuevas. |
| Potenciales | 19 % | 8 % | Desarrollar: promociones en los rubros que ya usan. |
| Ocasionales | 13 % | 3 % | Activación de bajo costo (app, cashback en la primera compra del mes). |
| En riesgo | 7 % | 8 % | **Primero el score.** Bajo riesgo: retención; alto riesgo: cobranza preventiva. |
| Enfriándose | 4 % | 1 % | Recordatorio barato antes de que pasen a Perdidos. |
| Perdidos | 16 % | 2 % | No gastar campaña en todos; reactivación solo a los de mejor score. |

### Puente al trabajo final

Las variables RFM son de las mejores para predecir **churn**: la recencia sola ya separa a los que se van. En el trabajo
final pueden construirlas desde `transacciones_mensual.csv`, **a la fecha de corte de cada periodo** (sin mirar el futuro,
como en la Demostración A.3).

---
# Cierre

### Los números del día

| | |
|---|---|
| Compras en la historia | 355 704 (julio 2025 a junio 2026), de 6 685 clientes |
| Fecha de corte | 30 de junio de 2026 |
| AUC de la recencia | 0.74 bien calculada · 0.89 con fuga de información |
| Concentración | Campeones + Leales: 43 % de clientes, 78 % del gasto |
| Volvieron a comprar | 97 % de los Campeones · 20 % de los Perdidos |
| Default | 31.9 % en En riesgo · 2.0 % en Campeones · cartera 9.1 % |

### Checklist de salida

- [ ] Construyo la tabla RFM desde el transaccional, **sin perder** a los clientes sin compras.
- [ ] Calculo todo a una **fecha de corte** y sé explicar por qué mirar el futuro infla el resultado.
- [ ] Uso quintiles con `rank(method="first")` y sé cuándo los quintiles no sirven (recencia muy concentrada).
- [ ] Armo un mapa de segmentos con reglas en orden y lo leo con el negocio (Pareto del gasto).
- [ ] Comparo reglas con K-Means y sé para qué sirve cada uno.
- [ ] Valido los segmentos con lo que pasó después y los cruzo con el riesgo antes de actuar.

### Preguntas tipo examen

**1.** Un analista calcula el RFM con `tx.groupby("id_cliente")` y reporta 6 365 clientes. La base tiene 6 685. ¿Qué pasó?

> **Respuesta.** El `groupby` sobre transacciones solo ve a los clientes que compraron. Los 320 sin compras en la ventana
> desaparecieron, y son justamente los inactivos. Hay que partir de la lista de clientes y completar con 0 compras.

**2.** La recencia calculada con todos los datos disponibles predice el default con AUC 0.89, y la calculada al corte con
0.74. ¿Cuál se usa?

> **Respuesta.** La de 0.74. La otra mira los meses en que se mide el default (cuando la tarjeta ya está bloqueada): es fuga
> de información. En producción no existe y el modelo se cae.

**3.** ¿Por qué se usa `rank(method="first")` antes de `pd.qcut` con la frecuencia?

> **Respuesta.** Porque la frecuencia es un conteo con muchos empates. Sin el ranking, `qcut` puede fallar por cortes
> repetidos o armar quintiles de tamaños muy distintos. El ranking rompe los empates y deja cinco grupos iguales.

**4.** El área comercial quiere mandar una oferta de aumento de línea a todo el segmento En riesgo. ¿Qué le dices?

> **Respuesta.** Que primero cruce con el score de riesgo: ese segmento tiene 31.9 % de default. Muchos dejaron de comprar
> porque coparon la línea. Ofrecerles más crédito aumenta la pérdida; a esos les corresponde cobranza preventiva.

**5.** K-Means sobre RFM y las reglas tienen un ARI de 0.25. ¿Alguno está mal?

> **Respuesta.** No necesariamente. Responden cosas distintas: las reglas son cortes de negocio fijos y explicables; K-Means
> agrupa por cercanía en los datos. Coinciden en los extremos y difieren en el medio. Se usa K-Means para explorar y las
> reglas para operar.

### Lo que sigue

Próxima clase: **puesta en producción e introducción a Databricks**. Y con este mismo transaccional, más adelante:
**fraude** (detección de anomalías) y **grafos** (quién está conectado con quién).